# Portable TRM GEMM Refiner

Clean Colab/T4 entrypoint for the Turing-first TRM GEMM prototype.

This notebook intentionally does three things in order:
1. install the project and Triton
2. run a strict preflight check and stop early if the environment is wrong
3. run a small real Triton-backed pipeline that should complete quickly on a T4


In [38]:
!git clone https://github.com/Cree0618/trm-youtubevids.git

Cloning into 'trm-youtubevids'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 46 (delta 8), reused 44 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 93.03 KiB | 807.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.


In [39]:
!git log

commit 46c2a4347dd0cb3dd7bc369d1e438bf79c90d612 (HEAD -> main, origin/main, origin/HEAD)
Author: Chris <142166970+Cree0618@users.noreply.github.com>
Date:   Wed Apr 1 18:28:55 2026 +0200

    Add portable TRM GEMM refiner

commit f967e0c4a14dd3a49f836c814ca6156f7aa2d517
Author: Chris <142166970+Cree0618@users.noreply.github.com>
Date:   Wed Apr 1 18:17:31 2026 +0200

    add trm

commit 429757205142ff270bc703c12c2044492e9b0b2a
Author: Chris <142166970+Cree0618@users.noreply.github.com>
Date:   Tue Mar 31 19:31:13 2026 +0200

    Create KV cache research brief

commit ea8a57ab38b579ee1dc4b89f5b556d6e769f32b7
Author: Chris <142166970+Cree0618@users.noreply.github.com>
Date:   Sat Mar 28 16:36:08 2026 +0100

    Initial video script and Manim baseline


In [35]:
# Set this if your repo is cloned somewhere else in Colab.
import os
from pathlib import Path

PROJECT_ROOT = Path('/content/youtube-videos')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

print('PROJECT_ROOT =', PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

PROJECT_ROOT = /content/trm-youtubevids


In [36]:
!pip uninstall -y trm-gemm >/dev/null 2>&1 || true
!pip install -e .
!pip install -r requirements-colab.txt

Obtaining file:///content/trm-youtubevids
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for trm-gemm (pyproject.toml) ... done
  Created wheel for trm-gemm: filename=trm_gemm-0.1.0-0.editable-py3-none-any.whl size=2708 sha256=75824e2e50bea933707c528b70f887755607574d22417de42cbf570361e2802a
  Stored in directory: /tmp/pip-ephem-wheel-cache-3xvjagd9/wheels/7c/10/ad/aa3047aeaf15cddb5bd07bf1c493df647fa66a18a67c001568
Successfully built trm-gemm


After the install cell finishes, restart the runtime once in Colab if this notebook had already imported `trm_gemm` earlier.

Then continue with the cells below.

In [37]:
from trm_gemm.env_check import print_preflight

preflight = print_preflight(strict_t4=True)
preflight

ModuleNotFoundError: No module named 'trm_gemm.env_check'

In [ ]:
!python -m trm_gemm.colab_smoke

In [ ]:
from trm_gemm.backend import TritonGemmBackend
from trm_gemm.baselines import greedy_search, random_search
from trm_gemm.data import TraceDataset, TraceGenerationConfig, generate_trace_records
from trm_gemm.model import TinyRecursiveGemmRefiner, rollout_refiner
from trm_gemm.training import compute_losses
from trm_gemm.types import GemmTaskSpec, T4

backend = TritonGemmBackend()
tasks = [
    GemmTaskSpec(256, 256, 256),
    GemmTaskSpec(512, 512, 512),
]
records = generate_trace_records(
    tasks,
    T4,
    backend,
    TraceGenerationConfig(seeds_per_task=1, max_steps_per_seed=1, invalid_samples_per_task=0),
)
dataset = TraceDataset(records)
print('num_records =', len(dataset))
print('first_record =', dataset[0].to_dict())

In [ ]:
model = TinyRecursiveGemmRefiner()
batch = records[: min(4, len(records))]
losses = compute_losses(model, batch)
print({k: round(v.item(), 4) for k, v in losses.items()})

In [ ]:
task = tasks[-1]
random_schedule, random_feedback = random_search(backend, task, T4, budget=4, seed=0)
greedy_schedule, greedy_feedback = greedy_search(backend, task, T4, budget=4)
edits = rollout_refiner(model, task, T4, random_schedule, random_feedback, steps=3)

print('random_schedule =', random_schedule.to_dict())
print('random_feedback =', random_feedback.to_dict())
print('greedy_schedule =', greedy_schedule.to_dict())
print('greedy_feedback =', greedy_feedback.to_dict())
print('rollout_edits =', [edit.to_dict() for edit in edits])